# PolarQuant Unified: Qwen3.8-27B (TODAS as Versões)

Um único notebook que quantiza o modelo em **4 variantes** e sobe cada uma para o seu HuggingFace.

Model: `Qwen/Qwen3.8-27B` — Qwen3.5 ForConditionalGeneration, denso 27B (48 Gated DeltaNet + 16 Full Attention head_dim=256), multimodal.

| # | Versão | Método | Bits/peso | Deploy | Repo HF (conta {HF_USER}) |
|---|---|---|---|---|---|
| 1 | **SDTP-Lossless** | Saliency + Dual trit-plane | ~4.43 | Transformers (reconstrução BF16) | `Qwen3.8-27B-SDTP-Lossless` |
| 2 | **HLWQ-Q5** | Hadamard + Lloyd-Max 5-bit | ~5.0 | polar_state codes | `Qwen3.8-27B-HLWQ-Q5` |
| 3 | **HLWQ-CT-INT4** | INT4 simétrico (group 128) | ~4.0 | vLLM / Marlin | `Qwen3.8-27B-HLWQ-CT-INT4` |
| 4 | **HLWQ-Q1** | Hadamard + Lloyd-Max 1-bit | ~1.125 | polar_state codes (extremo) | `Qwen3.8-27B-HLWQ-Q1` |

- Encoder visual, gates GDN (`in_proj_a/b`), embeddings e lm_head mantidos em **BF16** em todas as versões.
- KV: rotação Hadamard + quantização nas 16 camadas full-attn; estado Gated DeltaNet em BF16 (opcional no runtime).
- A lista `VERSIONS` é editável — rode só as que quiser.

**Requisitos de memória**: o modelo (~54 GB BF16) é recarregado por versão (evita 2x RAM). Colab com >56 GB RAM e bastante disco.


In [ ]:
# numpy/scipy juntos (ABI) - depois do restart NAO re-rode esta celula
!pip install -q --force-reinstall numpy==2.2.6 scipy
!pip install git+https://github.com/huggingface/transformers.git --force-reinstall -q
!pip install -q datasets accelerate safetensors sentencepiece tiktoken huggingface_hub tqdm requests Pillow psutil


print('\n' + '='*60)
print('*** RESTART RUNTIME NOW (Runtime -> Restart session) ***')
print('Then run Cell 2 onwards. Do NOT re-run Cell 1.')
print('='*60)


In [ ]:
import torch, transformers, psutil
print(f'transformers: {transformers.__version__}')
print(f'torch:        {torch.__version__}')
print(f'CPU RAM:      {psutil.virtual_memory().total/1e9:.0f} GB')
print(f'Disk free:    {psutil.disk_usage("/content").free/1e9:.0f} GB')
print(f'CUDA:         {torch.cuda.get_device_name(0)}' if torch.cuda.is_available() else 'No CUDA - CPU mode')


In [ ]:
import os, json
MODEL = 'Qwen/Qwen3.8-27B'
HF_USER = 'caiovicentino1'
ROOT = '/content/polarquant_qwen38'
HEAD_DIM = 256
NUM_LAYERS = 64
BS = 128
LAYER_TYPES = (['gdn','gdn','gdn','full_attn']) * 16
assert len(LAYER_TYPES) == NUM_LAYERS
os.makedirs(ROOT, exist_ok=True)


import torch
# ###### Detecção de GPU (RTX PRO 6000 Blackwell = 96 GB VRAM) ######
_CUDA=torch.cuda.is_available()
_AGB=torch.cuda.mem_get_info(0)[0]/1e9 if _CUDA else 0.0
GPU_OK = _CUDA and _AGB > 60.0   # =True se GPU consegue segurar o modelo BF16 (~54GB)
print(f'CUDA: {_CUDA}  //  VRAM disponível: {_AGB:.1f} GB  //  GPU_OK={GPU_OK}')
if _CUDA:
    print('  GPU:', torch.cuda.get_device_name(0))


# ###### Versões: edite esta lista ######
VERSIONS = [
    dict(key='sdtp_lossless', method='sdtp', bits=4.43, outlier=0.02, dual=True,
         repo='caiovicentino1/Qwen3.8-27B-SDTP-Lossless'),
    dict(key='hlwq_q5', method='hlwq', nbits=5, bits=5.0,
         repo='caiovicentino1/Qwen3.8-27B-HLWQ-Q5'),
    dict(key='ct_int4', method='ct', bits=4.0,
         repo='caiovicentino1/Qwen3.8-27B-HLWQ-CT-INT4'),
    dict(key='hlwq_q1', method='hlwq', nbits=1, bits=1.125,
         repo='caiovicentino1/Qwen3.8-27B-HLWQ-Q1'),
]
# Rode só um subconjunto com as keys:  RUN_KEYS=['sdtp_lossless']
RUN_KEYS = [v['key'] for v in VERSIONS]
print('Versões selecionadas:', RUN_KEYS)


# Medir baseline BF16? (1 recarga extra, importante p/ razão de perda)
EVAL_BASELINE = True
# Eval: mais chunks/len se estiver em GPU (rápido), menos se CPU.
EVAL_PPL = True
N_CHUNKS = 12 if GPU_OK else 3
MAX_LEN = 2048 if GPU_OK else 1024
DO_UPLOAD = True


In [ ]:
import torch, math, time, gc
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from scipy.stats import norm as sp_norm
from collections import defaultdict
from tqdm.auto import tqdm


# ########## Walsh-Hadamard ##########
def _build_H(n):
    if n == 1: return torch.tensor([[1.0]])
    h = _build_H(n // 2)
    return torch.cat([torch.cat([h,h],1), torch.cat([h,-h],1)],0)/math.sqrt(2)
H256 = _build_H(HEAD_DIM)
H128 = _build_H(BS)
print(f'H128 {tuple(H128.shape)}, H256 {tuple(H256.shape)}')


# ########## Lloyd-Max centroids ##########
_C={}
def get_centroids(bits):
    if bits in _C: return _C[bits]
    n=2**bits
    b=torch.linspace(-3.5,3.5,n+1); ct=(b[:-1]+b[1:])/2
    for _ in range(100):
        for i in range(n):
            a,c=b[i].item(),b[i+1].item(); mass=sp_norm.cdf(c)-sp_norm.cdf(a)
            if mass>1e-10: ct[i]=-(sp_norm.pdf(c)-sp_norm.pdf(a))/mass
        for i in range(1,n): b[i]=(ct[i-1]+ct[i])/2
    _C[bits]=ct.clone(); return _C[bits]


# ########## Generic bitpack (nbits 1..5) ##########
def bitpack(flat,nbits):
    flat=flat.long().reshape(-1); tot=flat.shape[0]
    if nbits==1:
        pad=(8-tot%8)%8
        if pad: flat=torch.cat([flat,torch.zeros(pad,dtype=flat.dtype)])
        c=flat.reshape(-1,8)
        b=c[:,0]|(c[:,1]<<1)|(c[:,2]<<2)|(c[:,3]<<3)|(c[:,4]<<4)|(c[:,5]<<5)|(c[:,6]<<6)|(c[:,7]<<7)
        return b.to(torch.uint8),tot
    elif nbits==2:
        pad=(4-tot%4)%4
        if pad: flat=torch.cat([flat,torch.zeros(pad,dtype=flat.dtype)])
        c4=flat.reshape(-1,4)
        return (c4[:,0]|(c4[:,1]<<2)|(c4[:,2]<<4)|(c4[:,3]<<6)).to(torch.uint8),tot
    elif nbits==3:
        pad=(8-tot%8)%8
        if pad: flat=torch.cat([flat,torch.zeros(pad,dtype=flat.dtype)])
        c=flat.reshape(-1,8)
        b0=(c[:,0]<<5)|(c[:,1]<<2)|(c[:,2]>>1)
        b1=((c[:,2]&1)<<7)|(c[:,3]<<4)|(c[:,4]<<1)|(c[:,5]>>2)
        b2=((c[:,5]&3)<<6)|(c[:,6]<<3)|c[:,7]
        return torch.stack([b0,b1,b2],-1).reshape(-1).to(torch.uint8),tot
    elif nbits==4:
        pad=(2-tot%2)%2
        if pad: flat=torch.cat([flat,torch.zeros(pad,dtype=flat.dtype)])
        c2=flat.reshape(-1,2)
        return ((c2[:,0]<<4)|c2[:,1]).to(torch.uint8),tot
    elif nbits==5:
        pad=(8-tot%8)%8
        if pad: flat=torch.cat([flat,torch.zeros(pad,dtype=flat.dtype)])
        c=flat.reshape(-1,8)
        b0=((c[:,0]<<3)|(c[:,1]>>2)).to(torch.uint8)
        b1=(((c[:,1]&3)<<6)|(c[:,2]<<1)|(c[:,3]>>4)).to(torch.uint8)
        b2=(((c[:,3]&15)<<4)|(c[:,4]>>1)).to(torch.uint8)
        b3=(((c[:,4]&1)<<7)|(c[:,5]<<2)|(c[:,6]>>3)).to(torch.uint8)
        b4=(((c[:,6]&7)<<5)|c[:,7]).to(torch.uint8)
        return torch.stack([b0,b1,b2,b3,b4],-1).reshape(-1),tot
    return flat.to(torch.uint8),tot


# ########## Skip patterns ##########
SKIP_PATTERNS=['visual','vision','multi_modal_projector','patch_embed','patch_conv',
    'norm','layernorm','rmsnorm','embed_tokens','lm_head','gate','router',
    'A_log','conv1d','dt_bias','linear_attn.in_proj_a','linear_attn.in_proj_b','mtp']
def should_skip(name):
    return any(p in name for p in SKIP_PATTERNS)


# ########## PolarQuant KV (full-attn rotated+quant, GDN passthrough) ##########
class BitPackerKV:
    @staticmethod
    def pack(codes,nbits):
        c=codes.long(); N=c.shape[0]
        if nbits==2:
            c=c.reshape(N,-1,4); return ((c[:,:,0]<<6)|(c[:,:,1]<<4)|(c[:,:,2]<<2)|c[:,:,3]).to(torch.uint8)
        elif nbits==3:
            c=c.reshape(N,-1,8)
            b0=(c[:,:,0]<<5)|(c[:,:,1]<<2)|(c[:,:,2]>>1)
            b1=((c[:,:,2]&1)<<7)|(c[:,:,3]<<4)|(c[:,:,4]<<1)|(c[:,:,5]>>2)
            b2=((c[:,:,5]&3)<<6)|(c[:,:,6]<<3)|c[:,:,7]
            return torch.stack([b0,b1,b2],-1).reshape(N,-1).to(torch.uint8)
        elif nbits==4: return ((c[:,0::2]<<4)|c[:,1::2]).to(torch.uint8)
        return codes.to(torch.uint8)
    @staticmethod
    def unpack(packed,nbits,D):
        p=packed.long(); N=p.shape[0]
        if nbits==2: return torch.stack([(p>>6)&3,(p>>4)&3,(p>>2)&3,p&3],-1).reshape(N,D)
        elif nbits==3:
            p3=p.reshape(N,-1,3); b0,b1,b2=p3[:,:,0],p3[:,:,1],p3[:,:,2]
            return torch.stack([(b0>>5)&7,(b0>>2)&7,((b0&3)<<1)|((b1>>7)&1),(b1>>4)&7,(b1>>1)&7,((b1&1)<<2)|((b2>>6)&3),(b2>>3)&7,b2&7],-1).reshape(N,D)
        elif nbits==4: return torch.stack([(p>>4)&15,p&15],-1).reshape(N,D)
        return p


class PolarQuantLayer:
    def __init__(self,nbits=3,residual_length=128,device='cpu'):
        self.nbits=nbits; self.residual_length=residual_length; self.device=device
        self.H=H256.to(device); self.scale=math.sqrt(HEAD_DIM)
        self._packed=None; self._norms=None; self._q_seq=0
        self._B=None; self._NH=None; self._D=None; self._can_quantize=True; self.residual=None
        self._ctv=None
    def _centroids(self):
        if self._ctv is None:
            n=2**self.nbits; lo=torch.linspace(-3.5,3.5,n+1); self._ctv=(lo[:-1]+lo[1:])/2
        return self._ctv
    def _quantize(self,t):
        flat=t.reshape(-1,HEAD_DIM).float(); norms=flat.norm(dim=1,keepdim=True).clamp(min=1e-10)
        rot=(flat/norms)@self.H*self.scale
        ct=self._centroids().to(flat.device)
        codes=(rot.unsqueeze(-1)-ct.view(1,1,-1)).abs().argmin(-1)
        return BitPackerKV.pack(codes.to(torch.uint8),self.nbits),norms.to(torch.bfloat16).squeeze(1)
    def _dequantize(self,packed,norms,B,H):
        codes=BitPackerKV.unpack(packed,self.nbits,HEAD_DIM)
        ct=self._centroids().to(packed.device)
        values=(ct[codes]/self.scale)
        values=(values@self.H)*norms.float().unsqueeze(1)
        S=packed.shape[0]//(B*H); return values.to(torch.bfloat16).reshape(B,H,S,HEAD_DIM)
    def update(self,nt):
        if self._B is None:
            self._B,self._NH=nt.shape[0],nt.shape[1]; self._D=nt.shape[3]; self._can_quantize=(self._D==HEAD_DIM)
        self.residual=nt if self.residual is None else torch.cat([self.residual,nt],2)
        if self._can_quantize and self.residual.shape[2]>self.residual_length:
            nq=self.residual.shape[2]-self.residual_length
            packed,norms=self._quantize(self.residual[:,:,:nq,:])
            self.residual=self.residual[:,:,nq:,:].contiguous()
            self._packed=packed if self._packed is None else torch.cat([self._packed,packed],0)
            self._norms=norms if self._norms is None else torch.cat([self._norms,norms],0)
            self._q_seq+=nq
        if self._packed is not None:
            return torch.cat([self._dequantize(self._packed,self._norms,self._B,self._NH),self.residual],2)
        return self.residual
    def get_seq_length(self):
        q=self._packed.shape[0]//(self._B*self._NH) if self._packed is not None and self._B and self._NH else 0
        return q+(self.residual.shape[2] if self.residual is not None else 0)
    def memory_bytes(self):
        t=self._packed.numel()+self._norms.numel()*2 if self._packed is not None else 0
        return t+(self.residual.numel()*2 if self.residual is not None else 0)


try:
    from transformers.cache_utils import LinearAttentionCacheLayerMixin
    _la_base=LinearAttentionCacheLayerMixin
except ImportError: _la_base=object
class _HybridCacheLayer(_la_base):
    def __init__(self): self.conv_states=None; self.recurrent_states=None
    def lazy_initialization(self,*a,**k): pass
    def update_conv_state(self,cs,**k): self.conv_states=cs; return cs
    def update_recurrent_state(self,rs,**k): self.recurrent_states=rs; return rs
from transformers.cache_utils import Cache
class PolarQuantKVCache(Cache):
    def __init__(self,num_layers,nbits=3,residual_length=128,device='cpu'):
        try: super().__init__()
        except (TypeError,ValueError): pass
        self.num_layers=num_layers; self.nbits=nbits
        self.kl=[PolarQuantLayer(nbits,residual_length,device) for _ in range(num_layers)]
        self.vl=[PolarQuantLayer(nbits,residual_length,device) for _ in range(num_layers)]
        self._seen_tokens=0
        if not hasattr(self,'layers'): self.layers=[None]*num_layers
    def update(self,ks,vs,layer_idx,cache_kwargs=None):
        if layer_idx==0: self._seen_tokens+=ks.shape[2]
        return self.kl[layer_idx].update(ks),self.vl[layer_idx].update(vs)
    def get_seq_length(self,layer_idx=0): return self.kl[layer_idx].get_seq_length()
    def get_max_cache_shape(self): return None
    def get_mask_sizes(self,q,layer_idx): return q,self.get_seq_length(layer_idx)
    def has_previous_state(self,layer_idx=0):
        has_kv=self.get_seq_length(layer_idx)>0
        has_linear=self.layers[layer_idx] is not None and isinstance(self.layers[layer_idx],_HybridCacheLayer)
        return has_kv or has_linear
    def update_conv_state(self,cs,layer_idx,**k):
        if self.layers[layer_idx] is None: self.layers[layer_idx]=_HybridCacheLayer()
        self.layers[layer_idx].conv_states=cs; return cs
    def update_recurrent_state(self,rs,layer_idx,**k):
        if self.layers[layer_idx] is None: self.layers[layer_idx]=_HybridCacheLayer()
        self.layers[layer_idx].recurrent_states=rs; return rs
    @property
    def seen_tokens(self): return self._seen_tokens
    @property
    def is_initialized(self): return self._seen_tokens>0
    def __getitem__(self,idx):
        k,v=self.kl[idx],self.vl[idx]; ko,vo=k.residual,v.residual
        if k._packed is not None and ko is not None:
            ko=torch.cat([k._dequantize(k._packed,k._norms,k._B,k._NH),ko],2)
            vo=torch.cat([v._dequantize(v._packed,v._norms,v._B,v._NH),vo],2)
        return (ko,vo)
    def __len__(self): return self.num_layers
    def __iter__(self):
        for i in range(self.num_layers): yield self[i]
    def memory_bytes(self):
        return sum(k.memory_bytes()+v.memory_bytes() for k,v in zip(self.kl,self.vl))
print('utils OK')


In [ ]:
from transformers import AutoTokenizer, AutoModelForImageTextToText, AutoModelForCausalLM, AutoModel


tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
print(f'Tokenizer: {tokenizer.__class__.__name__}, vocab={tokenizer.vocab_size}')


def load_model():
    t=time.time()
    m=None
    dm = 'cuda:0' if GPU_OK else 'cpu'   # segura modelo inteiro em GPU quando der
    for ln in ['AutoModelForImageTextToText','AutoModelForCausalLM','AutoModel']:
        try:
            loader={'AutoModelForImageTextToText':AutoModelForImageTextToText,
                    'AutoModelForCausalLM':AutoModelForCausalLM,'AutoModel':AutoModel}[ln]
            m=loader.from_pretrained(MODEL,dtype=torch.bfloat16,device_map=dm,
                                     trust_remote_code=True,low_cpu_mem_usage=True)
            print(f'Loaded via {ln} on {dm}: {m.__class__.__name__} in {time.time()-t:.0f}s'); break
        except Exception as e:
            print(f'  {ln} failed: {type(e).__name__}: {str(e)[:120]}')
    if m is None: raise RuntimeError('all loaders failed')
    m.eval(); return m


def enumerate_text_linears(m):
    lm={}; sk=[]
    for name,mod in m.named_modules():
        if isinstance(mod,nn.Linear):
            if should_skip(name): sk.append(name)
            else: lm[name]=mod
    return lm,sk


In [ ]:
import math
from datasets import load_dataset


def load_eval_chunks(n_chunks, max_len):
    try:
        ds=load_dataset('wikitext','wikitext-2-raw-v1',split='test')
        all_text='\n'.join(x['text'] for x in ds if len(x['text'].strip())>50)
        full=tokenizer(all_text,return_tensors='pt').input_ids[0]
    except Exception:
        full=tokenizer(('. '.join(['quantization preserves model quality through careful calibration and residual handling.']*80)),
                      return_tensors='pt',truncation=True,max_length=4096).input_ids[0]
    ch=[]
    for i in range(0,len(full)-1,max_len):
        c=full[i:i+max_len]
        if len(c)>=64: ch.append(c)
    return ch[:n_chunks]


EVAL_CHUNKS=[]
def get_chunks():
    global EVAL_CHUNKS
    if not EVAL_CHUNKS: EVAL_CHUNKS=load_eval_chunks(N_CHUNKS,MAX_LEN)
    return EVAL_CHUNKS


def compute_ppl(m, chunks):
    tl=0.0; tt=0; m.eval()
    dev=m.device if hasattr(m,'device') else next(m.parameters()).device
    with torch.no_grad():
        for ch in tqdm(chunks,desc='PPL'):
            ids=ch.unsqueeze(0).to(dev)
            if ids.shape[1]<2: continue
            out=m(input_ids=ids,labels=ids)
            n=ids.shape[1]-1; tl+=out.loss.item()*n; tt+=n
    return math.exp(tl/tt)


def gen_sanity(m, max_new=30):
    dev=m.device if hasattr(m,'device') else next(m.parameters()).device
    out_lines=[]
    for p in ['The capital of France is','def fibonacci(n):\n    if n < 2:\n        return n\n    return']:
        ids=tokenizer(p,return_tensors='pt').to(dev)
        with torch.no_grad():
            o=m.generate(**ids,max_new_tokens=max_new,do_sample=False,pad_token_id=tokenizer.eos_token_id)
        out_lines.append(tokenizer.decode(o[0],skip_special_tokens=True))
    return out_lines
print('eval helpers OK')


In [ ]:
# ########## SDTP ##########
def optimal_ternary_offset(w):
    mu=w.mean(); wc=w-mu; sig=wc.std()
    if sig<1e-10: return np.full_like(w,mu)
    tau=0.612*sig; alpha=1.224*sig; out=np.zeros_like(w)
    out[wc>tau]=alpha; out[wc<-tau]=-alpha; return out+mu


def sdtp_quantize(W,act_norm,outlier_frac,dual_plane):
    m,n=W.shape
    if act_norm is None or act_norm.shape[0]!=n: act_norm=np.ones(n,np.float32)
    saliency=np.abs(W)*act_norm[None,:]
    n_out=max(1,int(outlier_frac*W.size))
    thr=np.partition(saliency.flatten(),-n_out)[-n_out]; mask=saliency>=thr
    out=np.zeros_like(W)
    for i in range(m):
        rm=mask[i]
        if rm.any():
            vals=W[i][rm]; mv=np.abs(vals).max()+1e-10; sc=mv/127
            out[i,rm]=np.round(vals/sc).clip(-127,127)*sc
        nsm=~rm
        if nsm.any():
            v=W[i][nsm]; q1=optimal_ternary_offset(v)
            out[i,nsm]=q1+(optimal_ternary_offset(v-q1) if dual_plane else 0)
    return out


def capture_act_norms(m,lm):
    dev=m.device if hasattr(m,'device') else next(m.parameters()).device
    sumsq={}
    def mk(name):
        def hook(mod,inp,outp):
            x=inp[0].detach().float().reshape(-1,inp[0].shape[-1])
            s=(x**2).sum(0).cpu().numpy()
            sumsq[name]=sumsq.get(name,np.zeros_like(s))+s
        return hook
    hs=[mod.register_forward_hook(mk(n)) for n,mod in lm.items()]
    calib=['Machine learning models use gradient descent to minimize loss over training data.',
    'The mitochondria is the powerhouse of the cell, generating ATP via oxidative phosphorylation.',
    'def fibonacci(n):\n    if n < 2:\n        return n\n    return fibonacci(n-1)+fibonacci(n-2)']
    with torch.no_grad():
        for t in tqdm(calib,desc='Calib'):
            ids=tokenizer(t,return_tensors='pt',truncation=True,max_length=256).to(dev)
            try: m(**ids)
            except Exception: pass
    for h in hs: h.remove()
    return {n:np.sqrt(v+1e-10) for n,v in sumsq.items()}


def do_sdtp(m,lm,v):
    norms=capture_act_norms(m,lm)
    stats=defaultdict(lambda:{'mse':0.0,'cnt':0,'p':0})
    qp=0
    for name,mod in tqdm(lm.items(),desc='SDTP'):
        W=mod.weight.data.float().cpu().numpy()
        an=norms.get(name)
        Wq=sdtp_quantize(W,an,v['outlier'],v['dual'])
        key='.'.join(p for p in name.split('.')[-2:] if not p.isdigit())
        stats[key]['mse']+=float(np.mean((W-Wq)**2))*W.size; stats[key]['cnt']+=1; stats[key]['p']+=W.size
        mod.weight.data=torch.from_numpy(Wq).to(dtype=mod.weight.dtype,device=mod.weight.device)
        qp+=W.size; del W,Wq
    return {'quant_params':qp}


# ########## HLWQ (Hadamard + Lloyd-Max n-bit) ##########
def hlwq_quant2d(W,nbits,ct,H,dev):
    out_f,in_f=W.shape; pad=(BS-in_f%BS)%BS
    Wp=F.pad(W,(0,pad)) if pad else W
    nb=Wp.shape[1]//BS
    blocks=Wp.reshape(out_f,nb,BS).float()
    norms=blocks.norm(-1,keepdim=True).clamp(min=1e-10)
    blocks=blocks/norms
    rot=(blocks@H)*math.sqrt(BS)
    codes=(rot.unsqueeze(-1)-ct.view(1,1,1,-1)).abs().argmin(-1).to(torch.uint8)
    values=ct[codes.long()]/math.sqrt(BS)
    W_deq=((values@H.T)*norms).reshape(out_f,-1)[:,:in_f].to(torch.bfloat16)
    return W_deq,codes,norms


def do_hlwq(m,lm,v):
    dev=m.device if hasattr(m,'device') else next(m.parameters()).device
    nbits=v['nbits']
    ct=get_centroids(nbits).to(dev).float(); H=H128.to(dev).float()
    polar={}; qp=0
    for name,mod in tqdm(lm.items(),desc=f'HLWQ-Q{nbits}'):
        W=mod.weight.data.to(dev)
        W_deq,codes,norms=hlwq_quant2d(W,nbits,ct,H,dev)
        mod.weight.data=W_deq.to('cpu')
        packed,total=bitpack(codes.cpu().reshape(-1),nbits)
        sk=name.replace('.','__')
        out_f=W_deq.shape[0]; nb=W_deq.shape[1]//BS
        polar[f'{sk}__packed']=packed
        polar[f'{sk}__norms']=norms.half().cpu().reshape(out_f,nb)
        polar[f'{sk}__meta']=torch.tensor([out_f,nb,BS,total,mod.weight.shape[1]])
        qp+=W.numel(); del W,W_deq,codes,norms
    return {'polar':polar,'quant_params':qp}


# ########## CT INT4 (vLLM/Marlin) ##########
def int4_quant(W,gs=128):
    of,inf=W.shape; ng=inf//gs
    Wg=W.reshape(of,ng,gs).float()
    absmax=Wg.abs().amax(-1,keepdim=True).clamp(min=1e-10); sc=absmax/7.0
    Wi=(Wg/sc).round().clamp(-8,7).to(torch.int8)
    return Wi.reshape(of,inf),sc.squeeze(-1).to(torch.bfloat16)
def pack_int4(t):
    tt=(t.to(torch.int32)&15).reshape(*t.shape[:-1],-1,8); p=tt[...,0]
    for i in range(1,8): p=p|(tt[...,i]<<(4*i))
    return p
def to_ct_key(name): return name.replace('model.language_model.','model.')


def do_ct(m,lm,v):
    ct={}; bf16c=0; intc=0
    for name,mod in lm.items():
        W=mod.weight.data; cn=to_ct_key(name)
        if W.shape[1]%128!=0:
            ct[f'{cn}.weight']=W.cpu().to(torch.bfloat16); bf16c+=1; continue
        Wi,sc=int4_quant(W)
        ct[f'{cn}.weight_packed']=pack_int4(Wi).cpu()
        ct[f'{cn}.weight_scale']=sc.cpu()
        deq=(sc.unsqueeze(1)*Wi.float()).reshape(mod.weight.shape)
        mod.weight.data=deq.to(torch.bfloat16)
        intc+=1
    for name,param in m.named_parameters():
        if 'visual' in name or 'mtp' in name: continue
        cn=to_ct_key(name); base=cn.rsplit('.weight',1)[0] if cn.endswith('.weight') else cn
        if any(k.startswith(base+'.weight_packed') or k==cn for k in ct): continue
        ct[cn]=param.data.cpu().to(torch.bfloat16)
        bf16c+=1
    return {'tensors':ct,'int_count':intc,'bf16_count':bf16c}
print('quantizers OK')


In [ ]:
from safetensors.torch import save_file


results=[]
baseline_ppl=None


# baseline BF16 (deve vir antes das quantizadas p/ razão de perda)
if EVAL_BASELINE:
    print('\n[Baseline] medindo BF16 (sem quantização)...')
    _m=load_model(); baseline_ppl=compute_ppl(_m,get_chunks())
    print(f'  BASELINE PPL: {baseline_ppl:.4f}')
    del _m; gc.collect()


SELECTED=[v for v in VERSIONS if v['key'] in RUN_KEYS]
for vi,v in enumerate(SELECTED,1):
    key=v['key']
    vdir=f'{ROOT}/{key}'; os.makedirs(vdir,exist_ok=True)
    print('\n' + '='*72)
    print(f'[{vi}/{len(SELECTED)}] {key.upper()}  (bits~{v.get("bits")})  repo={v["repo"]}')
    print('='*72)
    m=load_model()
    lm,sk=enumerate_text_linears(m)
    print(f'  text linears: {len(lm)} quantize, {len(sk)} skip (BF16)')
    t0=time.time()
    row={'key':key,'repo':v['repo'],'bits':v.get('bits')}


    if v['method']=='sdtp':
        out=do_sdtp(m,lm,v); row['quant_params']=out['quant_params']; row['mode']='SDTP-Lossless'
    elif v['method']=='hlwq':
        out=do_hlwq(m,lm,v); row['quant_params']=out['quant_params']; row['mode']=f'HLWQ-Q{v["nbits"]}'
    elif v['method']=='ct':
        out=do_ct(m,lm,v); row['quant_params']=sum(t.numel() for t in out['tensors'].values()); row['mode']='CT-INT4'
    row['time']=time.time()-t0
    print(f'  quantizado em {row["time"]:.0f}s')


    if EVAL_PPL:
        try:
            row['ppl']=compute_ppl(m,get_chunks())
            if baseline_ppl: row['ratio']=row['ppl']/baseline_ppl
            print(f'  PPL: {row["ppl"]:.3f} (ratio {row.get("ratio",0):.3f}x)')
        except Exception as e:
            print(f'  PPL falhou: {type(e).__name__}: {str(e)[:80]}')
    try:
        gens=gen_sanity(m); row['sanity']=gens[0][:50]; print('  sanity:',gens[0][:60])
    except Exception as e:
        print(f'  sanity falhou: {type(e).__name__}')


    cfg=dict(m.config.to_dict()); cfg['quantization_config']={'quant_method':row['mode'],'source':MODEL}
    with open(f'{vdir}/config.json','w') as f: json.dump(cfg,f,indent=2)
    try: tokenizer.save_pretrained(vdir)
    except Exception: pass


    if v['method']=='sdtp':
        print('  salvando modelo reconstruído BF16...')
        m.save_pretrained(vdir)
    elif v['method']=='hlwq':
        print(f'  salvando {len(out["polar"])} tensores polar_state...')
        sh=[]; cur={}; sz=0; MAX=4*1024**3
        for k in sorted(out['polar']):
            t=out['polar'][k]; s=t.numel()*t.element_size()
            if sz+s>MAX and cur: sh.append(cur); cur={}; sz=0
            cur[k]=t; sz+=s
        if cur: sh.append(cur)
        for i,sc in enumerate(sh,1):
            save_file(sc,f'{vdir}/polar_state-{i:05d}-of-{len(sh):05d}.safetensors')
        row['size_gb']=sum(os.path.getsize(f'{vdir}/{f}') for f in os.listdir(vdir) if f.startswith('polar_state'))/1e9
        print(f'  polar_state: {row["size_gb"]:.2f} GB em {len(sh)} shards')
    elif v['method']=='ct':
        print(f'  salvando CT INT4 ({len(out["tensors"])} tensores)...')
        sh=[]; cur={}; sz=0; MAX=4*1024**3; wm={}
        for k in sorted(out['tensors']):
            t=out['tensors'][k]; s=t.numel()*t.element_size()
            if sz+s>MAX and cur: sh.append(cur); cur={}; sz=0
            cur[k]=t; sz+=s
        if cur: sh.append(cur)
        for i,sc in enumerate(sh,1):
            fn=f'model-{i:05d}-of-{len(sh):05d}.safetensors'; save_file(sc,f'{vdir}/{fn}')
            for k in sc: wm[k]=fn
        tot=sum(os.path.getsize(f'{vdir}/{f}') for f in os.listdir(vdir) if f.endswith('.safetensors'))
        with open(f'{vdir}/model.safetensors.index.json','w') as f:
            json.dump({'metadata':{'total_size':tot},'weight_map':wm},f)
        cc=dict(m.config.to_dict())
        cc['quantization_config']={'quant_method':'compressed-tensors','format':'pack-quantized',
            'config_groups':{'group_0':{'targets':['Linear'],'weights':{'num_bits':4,'type':'int','symmetric':True,'group_size':128,'strategy':'group'}}},
            'ignore':['lm_head','embed_tokens','re:.*layernorm.*','re:.*visual.*','re:.*vision.*','re:.*multi_modal_projector.*','re:.*A_log','re:.*conv1d.*','re:.*dt_bias','re:.*in_proj_a.*','re:.*in_proj_b.*']}
        with open(f'{vdir}/config.json','w') as f: json.dump(cc,f,indent=2)
        row['size_gb']=tot/1e9; print(f'  CT INT4: {tot/1e9:.2f} GB em {len(sh)} shards')
    results.append(row)
    del m; gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()


print('\nTodas as versões concluídas.')


In [ ]:
HF_TOKEN = ''  # @param {type:'string'} -- cole seu token do HF
if HF_TOKEN and DO_UPLOAD:
    import os as _os
    _os.environ['HF_TOKEN']=HF_TOKEN
    from huggingface_hub import HfApi
    api=HfApi()
    for r in results:
        vdir=f'{ROOT}/{r["key"]}'
        print(f'\nUpload {r["key"]} -> {r["repo"]}')
        try:
            api.create_repo(r['repo'],repo_type='model',exist_ok=True,private=False)
            api.upload_large_folder(folder_path=vdir,repo_id=r['repo'],repo_type='model')
            print(f'  OK https://huggingface.co/{r["repo"]}')
        except Exception as e:
            print(f'  FAIL: {type(e).__name__}: {str(e)[:120]}')
else:
    print('[!] Defina HF_TOKEN (e DO_UPLOAD=True) e re-execute esta célula para subir')


In [ ]:
print('\n' + '='*78)
print('RESUMO FINAL - TODAS AS VERSÕES  |  model='+MODEL)
print('='*78)
print(f"{'versão':<14}{'método':<16}{'bits':>6}{'PPL':>9}{'razão':>8}{'tempo':>8}{'sizeGB':>9}  HF repo")
for r in results:
    ppl=r.get('ppl','-'); ratio=r.get('ratio',0)
    print(f"{r['key']:<14}{r['mode']:<16}{r['bits']:>6}"+
          f"{str(round(ppl,3) if ppl!='-' else '-'):>9}"+
          f"{f'{ratio:.3f}x' if ratio else '-':>8}"+
          f"{round(r['time'],0):>8.0f}"+
          f"{str(r.get('size_gb','-')):>9}  https://huggingface.co/{r['repo']}")
if baseline_ppl:
    print(f"\nBaseline BF16 PPL: {baseline_ppl:.3f}  (referência p/ razões)")
print('\nDone!')
